# Interpreting the selected job-posting SVM

Which words and phrases push a posting's score toward **Fake** or **Real**? We fit the Step 2 configuration retained in Step 3 on the development postings. We inspect the raw SVM because these coefficients determine its score before Platt calibration.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from fakejob.data import load_split
from fakejob.features import STEP2_CONFIG, build_pipeline, posting_features

# Reconstruct the project's fixed split using the same scikit-learn version.
if sklearn.__version__ != "1.6.1":
    raise RuntimeError("Use scikit-learn 1.6.1 to reproduce the project split.")
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="IProgress not found.*")
    split = load_split()
dev = split.model_df.loc[split.dev_index]
assert len(dev) == 15001 and dev.index.intersection(split.test_index).empty
X = posting_features(dev)
y = dev["fraudulent"]
print(f"Development postings: {len(dev):,} ({y.sum():,} Fake, {(y == 0).sum():,} Real)")

Development postings: 15,001 (724 Fake, 14,277 Real)


In [2]:
# Fit all feature blocks together so the weights belong to the selected model.
model = build_pipeline(STEP2_CONFIG).fit(X, y)
transformer, svm = model.steps[0][1], model.steps[-1][1]
assert list(svm.classes_) == [0, 1]
word = transformer.named_transformers_["word"]
terms = word.get_feature_names_out()
weights = svm.coef_[0, transformer.output_indices_["word"]]
assert len(weights) == len(terms)
print(f"Word and phrase coefficients available: {len(terms):,}")

Word and phrase coefficients available: 273,800


## Words and phrases with the strongest weights

Each table lists the five highest and five lowest coefficients within its feature type. A positive coefficient increases the Fake score when the term is present. The contribution for one posting equals the coefficient multiplied by that posting's TF-IDF value. The decision also uses character fragments and structured fields. Counts show how many development postings of each class contain the term.

In [3]:
# Count each matching posting once per term using the fitted word vocabulary.
matrix = word.transform(X["combined_text"]).tocsc()
signals = pd.DataFrame({
    "Term": terms, "Weight": weights,
    "Fake postings": matrix[y.to_numpy() == 1].getnnz(axis=0),
    "Real postings": matrix[y.to_numpy() == 0].getnnz(axis=0),
})
for title, subset in signals.groupby(np.where(signals["Term"].str.contains(" "), "Two-word phrases", "Single words")):
    print(title)
    extremes = pd.concat([subset.nlargest(5, "Weight").assign(Direction="Fake"),
                          subset.nsmallest(5, "Weight").assign(Direction="Real")])
    display(extremes.set_index(["Direction", "Term"]).round(3))

Single words


Weight  Fake postings  Real postings
Direction Term                                                
Fake      northwestern     0.761              1              3
          rohan            0.667              3              0
          link             0.630             53            191
          administrative   0.578             94            453
          assistant        0.559             81            598
Real      it              -0.481            111           6415
          web             -0.432             24           2677
          our             -0.412            387          11130
          of              -0.347            567          13547
          companies       -0.340             85           3627

Two-word phrases


Weight  Fake postings  Real postings
Direction Term                                                      
Fake      using below            0.794             34              0
          below link             0.788             35              1
          apply using            0.746             30              2
          junior project         0.744              1              5
          executive sales        0.741              1              5
Real      will be               -0.398            135           5055
          is looking            -0.371             21           1304
          your resume           -0.339             29            452
          company in            -0.334              5            387
          sales representative  -0.318              2            305

## Reading the signals in context

Inspect application instructions, a rare name-like term, and a phrase associated with Real. For comparison, also look up two payment-related phrases suggested before this analysis. Group counts indicate whether matches are concentrated in similar descriptions. Excerpts use the first matching development posting in each class and show normalized tokens in their original order.

In [4]:
tokenize = word.build_tokenizer()
preprocess = word.build_preprocessor()
checks = ["below link", "rohan", "is looking", "registration fee", "wire transfer"]
groups = split.groups.loc[dev.index]
for term in checks:
    column = word.vocabulary_.get(term)
    if column is None:
        print(f"{term!r}: outside the fitted word vocabulary.")
        continue
    positions = matrix[:, column].nonzero()[0]
    print(f"{term!r} | weight {weights[column]:+.3f} | {len(positions)} postings | "
          f"{groups.iloc[positions].nunique()} description groups")
    phrase = term.split()
    for label, name in [(1, "Fake"), (0, "Real")]:
        matches = positions[y.iloc[positions].to_numpy() == label]
        if len(matches):
            pos = matches[0]
            tokens = tokenize(preprocess(X["combined_text"].iloc[pos]))
            start = next(i for i in range(len(tokens)) if tokens[i:i + len(phrase)] == phrase)
            excerpt = " ".join(tokens[max(0, start - 8):start + len(phrase) + 8])
            print(f"  {name} | job {dev['job_id'].iloc[pos]} | … {excerpt} …")

'below link' | weight +0.788 | 36 postings | 25 description groups
  Fake | job 585 | … quality improvement manager apply using below link url_8b28dea5804b323e29db29adaf9fa38b688e5d9db1f6cf5b6b2e924236a07272 the quality improvement manager is responsible for …
  Real | job 4811 | … the international graduates programme please click on the below link url_041db52b909170920e2c98ffe13587eb084c3d337b16a4f02f3ca33b6b398941 …
'rohan' | weight +0.667 | 3 postings | 3 description groups
  Fake | job 11038 | … migration center work with rohan to develop …
'is looking' | weight -0.371 | 1325 postings | 1086 description groups
  Fake | job 741 | … person that is team oriented and someone who is looking to share their knowledge while seeking new opportunities …
  Real | job 2 | … and excellent customer service our rapidly expanding business is looking for talented project manager to manage the successful …
'registration fee': outside the fitted word vocabulary.
'wire transfer' | weight -0.005 | 39 

## What the evidence supports

- **Application instructions.** “Using below”, “below link” and “apply using” can overlap in a single instruction. Their positive weights point to wording worth reviewing. A repeated template can produce all three signals.
- **Possible dataset shortcuts.** “Rohan” appears in only three development postings. A name-like token with a high weight is a candidate for a masking experiment. The coefficient alone cannot establish that the model memorizes employers.
- **Language associated with Real.** “Is looking” occurs in both classes. Its negative coefficient describes how it changes this model's score alongside its other inputs. It cannot establish that an individual posting is genuine.
- **Follow-up experiment.** Compare the selected SVM with and without company profiles on the same grouped development folds. This tests how much the model depends on that field. A separate name-masking comparison can isolate the effect of names more directly.

## How much does the SVM rely on company profiles?

We compare the published Step 2 SVM scores with a new run that blanks the company profile before building features. Titles, job descriptions, company-logo flags and the other structured inputs are still available. Both runs use the same development rows, description groups, seed and nested threshold-selection procedure. The held-out test set is excluded. The comparison measures the contribution of the entire profile field, including names, writing style and company information.

In [1]:
from pathlib import Path
import sklearn
from fakejob.data import load_split
from fakejob.evaluation import repeated_group_cv
from fakejob.features import STEP2_CONFIG, build_pipeline, posting_features

# This experiment can start in a fresh kernel with the project's pinned version.
if sklearn.__version__ != "1.6.1":
    raise RuntimeError("Select a kernel with scikit-learn 1.6.1 to reproduce the project split.")
split = load_split()
dev = split.model_df.loc[split.dev_index]
y = dev["fraudulent"]
assert len(dev) == 15001 and dev.index.intersection(split.test_index).empty

# Remove the raw text before building both word and character features.
without_profile = posting_features(dev.assign(company_profile=""))
assert without_profile["company_profile"].eq("").all()
assert without_profile.index.equals(y.index)
print(f"Profiles removed from {dev['company_profile'].fillna('').str.strip().ne('').sum():,} of {len(dev):,} development postings.")

# Repeat the exact Step 2 evaluation with the ablated inputs.
ablated = repeated_group_cv(
    build_pipeline(STEP2_CONFIG), without_profile, y, split.groups.loc[y.index],
    name="Without company profile", n_splits=5, n_repeats=3, inner_splits=3,
    seed=0, n_jobs=2,
)

# Save the completed run so the comparison can be opened in a new session.
fold_file = Path("results") / "interpretation_without_profile_folds.csv"
ablated.folds.to_csv(fold_file, index=False)
print(f"Saved {len(ablated.folds)} folds to {fold_file}.")

/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Profiles removed from 12,203 of 15,001 development postings.


/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Saved 15 folds to results/interpretation_without_profile_folds.csv.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Reuse the original scores from feature_engineering.ipynb as the paired baseline.
keys = ["repeat", "fold"]
baseline_folds = pd.read_csv(Path("results") / "step2_combined_folds.csv").set_index(keys).sort_index()
# Read both runs from disk so this cell works on its own.
ablated_folds = pd.read_csv(Path("results") / "interpretation_without_profile_folds.csv").set_index(keys).sort_index()
pd.testing.assert_frame_equal(baseline_folds[["n_test", "n_fake"]], ablated_folds[["n_test", "n_fake"]])
assert len(ablated_folds) == 15 and ablated_folds.index.is_unique

metrics = ["average_precision", "precision", "recall", "f1"]
change = ablated_folds[metrics] - baseline_folds[metrics]
summary = pd.DataFrame({
    "With profile": baseline_folds[metrics].mean(),
    "Without profile": ablated_folds[metrics].mean(),
    "Change": change.mean(),
    "Folds lower without profile": (change < -1e-12).sum(),
})
display(summary.rename(index={"average_precision": "PR-AUC (AP)", "precision": "Fake precision",
                             "recall": "Fake recall", "f1": "Fake F1"}).round(4))

# Each repeat evaluates every development posting once across its five folds.
def missed_by_repeat(folds):
    detected = (folds["n_fake"] * folds["recall"]).round().astype(int)
    return (folds["n_fake"] - detected).groupby(level="repeat").sum()

missed = pd.DataFrame({"Missed with profile": missed_by_repeat(baseline_folds),
                       "Missed without profile": missed_by_repeat(ablated_folds)})
missed["Additional misses"] = missed["Missed without profile"] - missed["Missed with profile"]
display(missed)

,With profile,Without profile,Change,Folds lower without profile
PR-AUC (AP),0.8705,0.7228,-0.1477,15
Fake precision,0.8337,0.7338,-0.0999,15
Fake recall,0.7520,0.6005,-0.1515,15
Fake F1,0.7894,0.6557,-0.1337,15


,Missed with profile,Missed without profile,Additional misses
repeat,,,
0,179,278,99
1,168,281,113
2,177,293,116


### What changed when the profile was removed?

Fake recall fell from **75.2% to 60.1%**, and PR-AUC fell from **0.871 to 0.723**. All 15 paired folds showed lower recall and PR-AUC. In each repeat of 724 fake postings, removing profiles led to **99, 113 and 116 additional misses**, respectively. Each repeat evaluates the same development postings, so these counts should be read separately.

The company profile supplies useful information on this dataset. Removing it simultaneously removes employer names, template wording and the distinction between filled and empty profiles. To test whether names specifically explain part of the gain, retain the profile text and mask names in a separate paired experiment.

## Does masking recognizable company names change performance?

This paired sensitivity check keeps the company profile. A fixed, label-independent rule recognizes capitalized names followed by a legal suffix such as “Inc”, “LLC” or “Ltd”. Within each posting it replaces the recognized name with a common token in the title, profile and description. Standalone occurrences of the name's stem are replaced too, unless the stem consists only of ordinary words (words written in lower case in at least 50 development postings, found without labels), so that a match such as “Recruit Ltd” does not remove every “recruit” from the posting. Detection is local to the posting. This rule can miss brands, aliases and lowercase names, or catch a third-party organization. Coverage is reported by class to show how much of the dataset this experiment actually changes.

In [1]:
from pathlib import Path
import re
import pandas as pd
import sklearn
from IPython.display import display
from fakejob.data import load_split
from fakejob.features import CORE_TEXT_FIELDS, STEP2_CONFIG, build_pipeline, posting_features
from fakejob.evaluation import repeated_group_cv

if sklearn.__version__ != "1.6.1":
    raise RuntimeError("Use scikit-learn 1.6.1 to reproduce the project split.")
split = load_split()
dev = split.model_df.loc[split.dev_index].copy()
y = dev["fraudulent"]
assert len(dev) == 15001 and dev.index.intersection(split.test_index).empty

columns = list(CORE_TEXT_FIELDS)
texts = dev[columns].fillna("")

# Ordinary vocabulary: words written in lower case in at least 50 development postings.
# Built without labels; a capitalised stem made only of such words ("Solutions", "Recruit",
# "Requires", "Time") is too generic to be replaced wherever it appears.
lowercase_words = texts.agg(" ".join, axis=1).str.findall(r"\b[a-z][a-z'’-]+\b").map(set)
COMMON_MIN_POSTINGS = 50
common = {word for word, n in lowercase_words.explode().value_counts().items() if n >= COMMON_MIN_POSTINGS}

token = r"[A-Z][A-Za-z0-9'’&-]*"
legal_name = re.compile(
    rf"\b(?P<stem>{token}(?:[ \t]+{token}){{0,4}}?)(?:,[ \t]*|[ \t]+)"
    r"(?i:incorporated|corporation|limited|corp|inc|llc|ltd|plc|llp|gmbh)\b"
)

def is_generic(stem):
    return all(word.lower() in common for word in stem.split())

def mask_names(fields):
    # The full matched name ("Recruit Ltd") is always replaced. The stem alone is replaced
    # elsewhere only when it is not made of ordinary words, so "recruit" in a description stays.
    names, stems = set(), set()
    for field in fields:
        for match in legal_name.finditer(field):
            names.add(match.group())
            if not is_generic(match["stem"]):
                stems.add(match["stem"])
    if not names:
        return fields, []
    variants = sorted(names | stems, key=lambda text: (-len(text), text))
    pattern = re.compile(r"(?<!\w)(?:" + "|".join(map(re.escape, variants)) + r")(?!\w)", re.I)
    return [pattern.sub("ORGNAME", field) for field in fields], sorted(names)

masked = dev.copy()
changed, examples, skipped = [], [], set()
for idx, fields in zip(dev.index, texts.itertuples(index=False, name=None)):
    replacements, names = mask_names(fields)
    if names:
        masked.loc[idx, columns] = replacements
        changed.append(idx)
        skipped |= {m["stem"] for f in fields for m in legal_name.finditer(f) if is_generic(m["stem"])}
        if len(examples) < 5:
            examples.append({"Job ID": dev.at[idx, "job_id"], "Matched text": names[0]})
coverage = pd.crosstab(y, dev.index.isin(changed)).reindex(columns=[False, True], fill_value=0)
coverage.columns = ["Unchanged", "Masked"]
coverage["Masked %"] = 100 * coverage["Masked"] / coverage.sum(axis=1)
display(coverage.rename(index={0: "Real", 1: "Fake"}).round(1))
display(pd.DataFrame(examples))
print(f"{len(common):,} ordinary words. Stems replaced only as part of the full name "
      f"({len(skipped)}), for example: {', '.join(sorted(skipped)[:12])}")

# Only development data is used. Preserve the baseline ordering, folds and threshold policy.
result = repeated_group_cv(
    build_pipeline(STEP2_CONFIG), posting_features(masked), y, split.groups.loc[y.index],
    name="Masked company names", n_splits=5, n_repeats=3, inner_splits=3, seed=0, n_jobs=3,
)
fold_file = Path("results") / "interpretation_masked_names_folds.csv"
assert len(result.folds) == 15
result.folds.to_csv(fold_file, index=False)
print(f"Saved {len(result.folds)} folds to {fold_file}.")

C:\Users\msi-nb\OneDrive\Desktop\ml-se3513\SE3513-ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Unchanged,Masked,Masked %
fraudulent,,,
Real,13143,1134,7.9
Fake,687,37,5.1


,Job ID,Matched text
0,6,"Apex Companies, LLC"
1,20,DCS PLC
2,26,Technologies Corporation
3,62,Lesaffre Yeast Corporation
4,76,Lesson Nine GmbH


3,870 ordinary words. Stems replaced only as part of the full name (52), for example: AM, Advertising, At The, Big, Boat International Media, But Not, Climate, Consulting, Create More, Cultural Service, DEEP, DUTIES INCLUDE BUT ARE NOT


Saved 15 folds to results\interpretation_masked_names_folds.csv.


In [1]:
# Run this cell independently to compare the three saved experiments.
from pathlib import Path
import pandas as pd
from IPython.display import display

files = {"Original": "step2_combined_folds.csv",
         "Names masked": "interpretation_masked_names_folds.csv",
         "Profile removed": "interpretation_without_profile_folds.csv"}
runs = {name: pd.read_csv(Path("results") / file).set_index(["repeat", "fold"]).sort_index()
        for name, file in files.items()}
original = runs["Original"]
assert len(original) == 15 and original.index.is_unique
metrics = ["average_precision", "precision", "recall", "f1"]
for run in runs.values():
    pd.testing.assert_frame_equal(run[["n_test", "n_fake"]], original[["n_test", "n_fake"]])
table = pd.DataFrame({name: run[metrics].mean() for name, run in runs.items()})
diff = runs["Names masked"][metrics] - original[metrics]
table["Masked minus original"] = diff.mean()
table["Folds lower after masking"] = (diff < -1e-12).sum()
display(table.rename(index={"average_precision": "PR-AUC (AP)", "precision": "Fake precision",
                            "recall": "Fake recall", "f1": "Fake F1"}).round(4))

,Original,Names masked,Profile removed,Masked minus original,Folds lower after masking
PR-AUC (AP),0.8705,0.8712,0.7228,0.0007,2
Fake precision,0.8337,0.8211,0.7338,-0.0126,6
Fake recall,0.7520,0.7569,0.6005,0.0049,3
Fake F1,0.7894,0.7859,0.6557,-0.0035,8


### Interpretation

The rule changed 1,171 of 15,001 development postings, including 37 of 724 fake postings. Average precision stayed close to its original value (0.8705 before masking and 0.8712 after); fake F1 moved from 0.7894 to 0.7859. In contrast, removing the whole profile reduced fake F1 to 0.6557. The profile-removal loss was not reproduced by masking this subset of organization mentions. The experiment leaves unrecognized names, company-specific wording and template reuse unresolved.

## Experiment: SVM errors with empty and filled company profiles

For the first repeat of Step 2, score each posting once in its held-out description group. Use the original inner-validation threshold for each fold and check the reconstructed metrics against the saved fold scores. Count whitespace-only and missing profiles as empty. This measures performance on naturally occurring subgroups.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold
from fakejob.data import load_split
from fakejob.evaluation import all_metrics
from fakejob.features import STEP2_CONFIG, build_pipeline, posting_features

if sklearn.__version__ != "1.6.1":
    raise RuntimeError("Use scikit-learn 1.6.1 for the saved Step 2 folds.")
split = load_split()
dev = split.model_df.loc[split.dev_index]
X = posting_features(dev)
y = dev.fraudulent
groups = split.groups.loc[dev.index]
results = Path("results")
reference = pd.read_csv(results / "step2_combined_folds.csv").query("repeat == 0").set_index("fold")
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
batches = []
for fold, (train, valid) in enumerate(cv.split(X, y, groups)):
    assert set(groups.iloc[train]).isdisjoint(groups.iloc[valid])
    model = build_pipeline(STEP2_CONFIG).fit(X.iloc[train], y.iloc[train])
    scores = model.decision_function(X.iloc[valid])
    threshold = reference.loc[fold, "threshold"]
    metrics = all_metrics(y.iloc[valid], scores, threshold)
    np.testing.assert_allclose(list(metrics.values()), reference.loc[fold, list(metrics)], atol=1e-8)
    rows = dev.iloc[valid]
    batches.append(pd.DataFrame({
        "job_id": rows.job_id, "title": rows.title, "fold": fold,
        "profile": np.where(rows.company_profile.fillna("").str.strip().eq(""), "Empty", "Filled"),
        "actual": y.iloc[valid], "predicted": (scores >= threshold).astype(int),
        "margin_from_threshold": scores - threshold,
    }))

audit = pd.concat(batches)
assert audit.index.is_unique and audit.index.sort_values().equals(dev.index.sort_values())
assert audit.index.intersection(split.test_index).empty
audit.sort_values("job_id").to_csv(results / "step2_development_predictions_repeat0.csv", index=False)
print(f"Verified {len(audit):,} predictions against all five saved folds.")

/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Verified 15,001 predictions against all five saved folds.


In [1]:
# View the saved subgroup results and illustrative errors without retraining.
from pathlib import Path
import pandas as pd
from IPython.display import display

audit = pd.read_csv(Path("results") / "step2_development_predictions_repeat0.csv")
counts = audit.groupby(["profile", "actual", "predicted"]).size().unstack([1, 2], fill_value=0)
summary = pd.DataFrame({"Real": counts[0].sum(axis=1), "Fake": counts[1].sum(axis=1),
                        "Detected": counts[1, 1], "Missed": counts[1, 0], "False alarms": counts[0, 1]})
summary.loc["All"] = summary.sum()
summary["Fake recall"] = summary["Detected"] / summary["Fake"]
summary["False alarm rate"] = summary["False alarms"] / summary["Real"]
display(summary.round(4))

# Show a middle-ranked error by distance from the threshold for each type and subgroup.
errors = audit[audit.actual != audit.predicted].copy()
errors["Error"] = errors.actual.map({1: "Missed fake", 0: "False alarm"})
for (profile, kind), group in errors.groupby(["profile", "Error"]):
    ordered = group.sort_values(["margin_from_threshold", "job_id"])
    row = ordered.iloc[len(ordered) // 2]
    print(f"{profile} | {kind} | job {row.job_id} | {row.title}")

,Real,Fake,Detected,Missed,False alarms,Fake recall,False alarm rate
profile,,,,,,,
Empty,2314,484,333,151,83,0.6880,0.0359
Filled,11963,240,212,28,7,0.8833,0.0006
All,14277,724,545,179,90,0.7528,0.0063


Empty | False alarm | job 4657 | CALL CENTER MANAGER
Empty | Missed fake | job 8028 | Well Test Operator
Filled | False alarm | job 4673 | Operations/ Admin Supervisor / Awesome Person Wanted ASAP!
Filled | Missed fake | job 17686 | CUSTOMER SERVICE AGENT 


### What did the model miss?

The SVM caught 333 of 484 fake postings with an empty profile (68.8%) and 212 of 240 with a filled profile (88.3%). It also flagged 83 of 2,314 real postings with an empty profile and 7 of 11,963 with a filled profile. These groups differ in the postings they contain. The 19.5 percentage-point recall gap describes where errors concentrate. The earlier profile-removal experiment measures what happens when the input is changed.

## Step 4: DistilBERT fine-tuning pilot

Compare the selected SVM with a text-only DistilBERT on the same grouped development split. Both models train on the same examples, choose their fake-F1 threshold on a separate calibration subset and are scored on a predefined grouped subset of the original validation split. The project test set is excluded. This single-split pilot is exploratory; the 15-fold model-selection rule requires repeated grouped evaluation.

The SVM uses its selected text and structured features. DistilBERT reads the title, profile and description with separate token budgets. For a manageable local run we update its last transformer block and classification head for one epoch. Training-class frequencies determine the loss weights. The final results cell reads a CSV and can run by itself.

**Low-load settings.** CPU only, one numerical thread, one epoch, mini-batches of eight and at most 64 tokens per posting. Fixed grouped folds select approximately one eighth of the fitting pool, one quarter of the calibration pool and one quarter of the validation pool. Both models use these same subsets. Small fake-class counts and the short texts limit generalization. Use the saved comparison cell to view results without training again.

In [1]:
# Reproduce the small CPU pilot. Use the next cell to read the saved scores.
from fakejob.distilbert_pilot import run_pilot

run_pilot(use_gpu=False)

/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fit: 1,144 postings, 33 fake
Threshold selection: 600 postings, 27 fake
Evaluation: 483 postings, 47 fake


W1126 21:55:59.011000 22890 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8233.97it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fine-tuning on cpu


Training batch 100/143


Training batch 143/143


Saved both evaluation rows.


In [1]:
# Run this cell on its own to see the completed pilot.
from pathlib import Path
import pandas as pd
from IPython.display import display

pilot = pd.read_csv(Path("results") / "step4_distilbert_pilot.csv").set_index("model")
display(pilot[["average_precision", "precision", "recall", "f1",
               "detected", "missed", "false_alarms", "evaluation_size"]].round(4))

,average_precision,precision,recall,f1,detected,missed,false_alarms,evaluation_size
model,,,,,,,,
Selected SVM,0.8388,0.8667,0.8298,0.8478,39,8,6,483
DistilBERT (last block),0.1813,0.2292,0.2340,0.2316,11,36,37,483


### Pilot result

On the same 483 validation postings (47 fake), the SVM detected 39 fake postings with 6 false alarms. The CPU DistilBERT pilot detected 11 with 37 false alarms. This run supports retaining the selected SVM. Only 33 fake postings entered training. Short inputs, one epoch, partial fine-tuning and a single grouped split limit what this says about a fully trained transformer. These scores belong to this pilot and should be reported separately from the 15-fold development results.

## DistilBERT with more training postings on the Apple GPU

The larger run takes the first four of the original eight grouped training folds, including the original pilot's fold. The 600 threshold-selection and 483 evaluation postings stay identical. Both models are fitted again. DistilBERT keeps the same one epoch, eight-posting batches, short inputs, class weighting and trainable last block and head. Training pauses between GPU batches to reduce sustained load. The project test set remains held out.

In [1]:
# This cell retrains both models. The results cell below can run on its own.
from fakejob.distilbert_pilot import run_pilot

run_pilot(use_gpu=True)

/private/tmp/se3513-check.1ViDI3ND/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fit: 4,675 postings, 213 fake
Threshold selection: 600 postings, 27 fake
Evaluation: 483 postings, 47 fake


W1126 22:01:08.549000 24773 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8061.32it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (4 outputs left out)

Training batch 400/585


Training batch 500/585


Training batch 585/585


Saved both evaluation rows.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

results_dir = Path("results")
runs = {"Small CPU pilot": "step4_distilbert_pilot.csv",
        "Larger training split": "step4_distilbert_moderate.csv"}
table = pd.concat({name: pd.read_csv(results_dir / file).set_index("model")
                   for name, file in runs.items()}, names=["Run", "Model"])
display(table[["average_precision", "precision", "recall", "f1", "detected",
               "missed", "false_alarms", "evaluation_size"]].round(4))

average_precision  precision  \
Run                   Model                                                   
Small CPU pilot       Selected SVM                        0.8388     0.8667   
                      DistilBERT (last block)             0.1813     0.2292   
Larger training split Selected SVM                        0.9703     0.7458   
                      DistilBERT (last block)             0.6129     0.3922   

                                               recall      f1  detected  \
Run                   Model                                               
Small CPU pilot       Selected SVM             0.8298  0.8478        39   
                      DistilBERT (last block)  0.2340  0.2316        11   
Larger training split Selected SVM             0.9362  0.8302        44   
                      DistilBERT (last block)  0.4255  0.4082        20   

                                               missed  false_alarms  \
Run                   Model                                           
Small CPU pilot       Selected SVM                  8             6   
                      DistilBERT (last block)      36            37   
Larger training split Selected SVM                  3            15   
                      DistilBERT (last block)      27            31   

                                               evaluation_size  
Run                   Model                                     
Small CPU pilot       Selected SVM                         483  
                      DistilBERT (last block)              483  
Larger training split Selected SVM                         483  
                      DistilBERT (last block)              483

### What changed with more training data?

The larger training subset contained 4,675 postings, including 213 fake, up from 1,144 postings and 33 fake. On the same 47 fake validation postings, DistilBERT detected 20 rather than 11, while false alarms fell from 37 to 31. The SVM trained on the larger subset detected 44 with 15 false alarms. The larger pilot supports retaining the selected SVM. These are single-split development results; the comparison does not estimate performance on the project test set.